# MedTube Segmentation — RGBD Training (4-channel) on Colab GPU

Trains YOLO26n-seg on 4-channel RGB-D images (RGB + depth grayscale as 4th channel).

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 or A100)
2. Upload the `rgbd/` dataset folder to Google Drive
3. Update the `DATA_YAML` path below

In [ ]:
# Cell 1 — Install
!pip install ultralytics -q

In [ ]:
# Cell 2 — Check GPU
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 4 — Upload dataset
# Option A: Upload rgbd.zip to Drive, then unzip here
# !unzip /content/drive/MyDrive/medtube_rgbd.zip -d /content/rgbd

# Option B: Upload directly from local machine
# (run this, then drag-drop the rgbd/ folder)
import os
os.makedirs('/content/rgbd/images', exist_ok=True)
os.makedirs('/content/rgbd/labels', exist_ok=True)
print('Upload your rgbd images and labels, or unzip from Drive above')

In [ ]:
# Cell 5 — Create data.yaml for 4-channel RGBD
data_yaml = """train: /content/rgbd/images
val: /content/rgbd/images
test: /content/rgbd/images

nc: 4
names: ['Other', 'Push-on', 'Screwcap', 'Universal']
"""

with open('/content/rgbd/data.yaml', 'w') as f:
    f.write(data_yaml)

print('data.yaml created')
print(f"Images: {len(os.listdir('/content/rgbd/images'))}")
print(f"Labels: {len(os.listdir('/content/rgbd/labels'))}")

In [ ]:
# Cell 6 — Patch YOLO model for 4-channel input (ch=4)
# YOLO expects 3-channel RGB by default. We modify the first conv layer
# to accept 4 channels, keeping pretrained weights for RGB and adding
# a zero-initialized 4th channel.

from ultralytics import YOLO
import torch
import copy

model = YOLO('yolo11n-seg.pt')  # Using YOLO11n as base (smaller, faster)

# Get the first conv layer
first_conv = model.model.model[0].conv
old_weight = first_conv.weight.data  # shape: [out_ch, 3, kH, kW]

# Create new conv with 4 input channels
new_conv = torch.nn.Conv2d(
    4, old_weight.shape[0],
    kernel_size=first_conv.kernel_size,
    stride=first_conv.stride,
    padding=first_conv.padding,
    bias=first_conv.bias is not None,
)

# Copy RGB weights, zero-init the depth channel
with torch.no_grad():
    new_conv.weight[:, :3, :, :] = old_weight
    new_conv.weight[:, 3:, :, :] = 0.0  # depth channel starts at zero
    if first_conv.bias is not None:
        new_conv.bias = copy.deepcopy(first_conv.bias)

# Replace in model
model.model.model[0].conv = new_conv
model.model.model[0].conv.in_channels = 4

# Update model yaml to reflect 4 channels
if hasattr(model.model, 'yaml'):
    model.model.yaml['ch'] = 4

print(f'First conv patched: {first_conv.in_channels}ch → 4ch')
print(f'Weight shape: {new_conv.weight.shape}')

In [ ]:
# Cell 7 — Train
DRIVE_SAVE = '/content/drive/MyDrive/medtube_runs'

results = model.train(
    data     = '/content/rgbd/data.yaml',
    epochs   = 100,
    imgsz    = 640,
    batch    = 16,
    patience = 20,
    workers  = 2,
    device   = 0,
    project  = DRIVE_SAVE,
    name     = 'YOLO11n-RGBD',
    exist_ok = True,
    # Mild augmentation
    hsv_h=0.01, hsv_s=0.4, hsv_v=0.2,
    degrees=20.0, translate=0.05, scale=0.25,
    shear=2.0, perspective=0.0001,
    flipud=0.0, fliplr=0.5,
    mosaic=0.2, erasing=0.2,
    auto_augment='randaugment',
)

print(f'\nBest weights: {DRIVE_SAVE}/YOLO11n-RGBD/weights/best.pt')

In [ ]:
# Cell 8 — Evaluate
best = YOLO(f'{DRIVE_SAVE}/YOLO11n-RGBD/weights/best.pt')
r = best.val(data='/content/rgbd/data.yaml', split='test', imgsz=640, device=0)
print(f'Box  mAP50={r.box.map50:.4f}  mAP50-95={r.box.map:.4f}')
print(f'Mask mAP50={r.seg.map50:.4f}  mAP50-95={r.seg.map:.4f}')

In [ ]:
# Cell 9 — Download
from google.colab import files
files.download(f'{DRIVE_SAVE}/YOLO11n-RGBD/weights/best.pt')